In [1]:
# 화자 분리
import os


from dotenv import load_dotenv


load_dotenv()


HUGGING_FACE_TOKEN = os.getenv('HUGGING_FACE_TOKEN')


# print(HUGGING_FACE_TOKEN)


In [2]:
import warnings
warnings.filterwarnings('ignore')
from pyannote.audio import Pipeline # 화자 분리


In [3]:
# 모델 다운로드 후 로딩(메모리: 오브젝트 생성=> 실행하기 위해 생성)
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    use_auth_token=HUGGING_FACE_TOKEN
)


In [4]:
# pyannote는 torch를 의존함.(pyannote3.1.1이면 torch2.5.1이어야.)

import torch


if torch.cuda.is_available():   # GPU 사용 여부 확인
    pipeline.to(torch.device('cuda')) # model을 GPU로 이동
    print('Cuda is available')
else:
    print('Cuda is not available')


Cuda is available


In [5]:
diarization = pipeline('../audio/싼기타_비싼기타.mp3')


with open('싼기타_비싼기타.rttm', 'w', encoding='utf-8') as rttm:
    diarization.write_rttm(rttm)


In [6]:
# RTTM 파일 읽어서 CSV 변환
import pandas as pd


rttm_path = './싼기타_비싼기타.rttm'


df_rttm = pd.read_csv(
    rttm_path,
    sep=' ',
    header=None,
    names=['type','file','chnl','start','duration','C1','C2','speaker_id','C3','C4']
)
display(df_rttm)


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN
84,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN
85,SPEAKER,싼기타_비싼기타,1,423.642,0.781,NaN,NaN,SPEAKER_00,NaN,NaN
86,SPEAKER,싼기타_비싼기타,1,424.745,3.531,NaN,NaN,SPEAKER_00,NaN,NaN


In [7]:
# end 컬럼 생성
df_rttm["end"] = df_rttm["start"]+df_rttm["duration"]
df_rttm.head()

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN,11.384
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN,16.681
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN,27.886
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195


In [8]:
df_rttm["number"] = None    # number 컬럼 만든 후 None으로 초기화
df_rttm.at[0, "number"] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799,0
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN,11.384,None
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN,16.681,None
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN,27.886,None
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195,None
...,...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN,417.445,None
84,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN,421.230,None
85,SPEAKER,싼기타_비싼기타,1,423.642,0.781,NaN,NaN,SPEAKER_00,NaN,NaN,424.423,None
86,SPEAKER,싼기타_비싼기타,1,424.745,3.531,NaN,NaN,SPEAKER_00,NaN,NaN,428.276,None


In [9]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]


display(df_rttm.head(10))


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799,0
1,SPEAKER,싼기타_비싼기타,1,7.394,3.990,NaN,NaN,SPEAKER_01,NaN,NaN,11.384,0
2,SPEAKER,싼기타_비싼기타,1,11.740,4.941,NaN,NaN,SPEAKER_01,NaN,NaN,16.681,0
3,SPEAKER,싼기타_비싼기타,1,17.224,10.662,NaN,NaN,SPEAKER_01,NaN,NaN,27.886,0
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195,0
5,SPEAKER,싼기타_비싼기타,1,32.419,0.764,NaN,NaN,SPEAKER_00,NaN,NaN,33.183,1
6,SPEAKER,싼기타_비싼기타,1,33.557,3.548,NaN,NaN,SPEAKER_00,NaN,NaN,37.105,1
7,SPEAKER,싼기타_비싼기타,1,37.632,3.769,NaN,NaN,SPEAKER_00,NaN,NaN,41.401,1
8,SPEAKER,싼기타_비싼기타,1,41.621,0.017,NaN,NaN,SPEAKER_00,NaN,NaN,41.638,1
9,SPEAKER,싼기타_비싼기타,1,41.638,0.815,NaN,NaN,SPEAKER_01,NaN,NaN,42.453,2


In [ ]:
# df_rttm_grouped=df_rttm.groupby("number")     # grouping 결과만 나옴. 처리 필요
df_rttm_grouped=df_rttm.groupby("number").agg(  # groupby한 number는 index로 들어감.
    start=pd.NamedAgg(column="start", aggfunc=min),
    end=pd.NamedAgg(column="end", aggfunc=max),
    speaker_id=pd.NamedAgg(column="speaker_id", aggfunc="first")
)
df_rttm_grouped

,start,end,speaker_id
number,,,
0,0.976,30.195,SPEAKER_01
1,32.419,41.638,SPEAKER_00
2,41.638,42.453,SPEAKER_01
3,41.655,42.725,SPEAKER_00
4,42.674,44.032,SPEAKER_01
5,45.798,67.105,SPEAKER_00
6,67.224,82.793,SPEAKER_01
7,84.643,102.572,SPEAKER_00
8,103.489,117.530,SPEAKER_01


In [13]:
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
df_rttm_grouped

,start,end,speaker_id,duration
0,0.976,30.195,SPEAKER_01,29.219
1,32.419,41.638,SPEAKER_00,9.219
2,41.638,42.453,SPEAKER_01,0.815
3,41.655,42.725,SPEAKER_00,1.070
4,42.674,44.032,SPEAKER_01,1.358
5,45.798,67.105,SPEAKER_00,21.307
6,67.224,82.793,SPEAKER_01,15.569
7,84.643,102.572,SPEAKER_00,17.929
8,103.489,117.530,SPEAKER_01,14.041
9,119.737,138.667,SPEAKER_00,18.930


In [ ]:
# df_rttm_grouped.index.name=None   # reset_index와 동일?

In [14]:
# csv 파일 생성
df_rttm_grouped.to_csv(
    '싼기타_비싼기타.csv',
    sep=',',
    index=False
)
